In [7]:
import pandas as pd

# 1. LOAD THE DATA

In [8]:
df = pd.read_csv('data/preprocessed/cleaned_retail_data_2022_2025.csv')
rfm = pd.read_csv('data/preprocessed/rfm_base.csv', index_col='CustomerName')
item_similarity_df = pd.read_pickle('data/preprocessed/item_similarity_matrix.pkl')

b2b_rules = pd.read_csv('data/preprocessed/b2b_association_rules.csv') 
b2c_rules = pd.read_csv('data/preprocessed/b2c_association_rules.csv')

popularity = df.groupby('ProductName')['InvoiceID'].nunique().reset_index()
popularity.columns = ['ProductName', 'PurchaseFrequency']
popularity = popularity.sort_values(by='PurchaseFrequency', ascending=False)

# 2. HYBRID RECOMMENDATION

In [ ]:
def get_hybrid_recommendations(customer_name, current_basket):
    """
    Combines CF, Association Rules, and Popularity based on customer type.
    - B2B (known contractors): CF (item similarity) + Association Rules
    - B2C (anonymous walk-ins): Association Rules only
    - Falls back to store best-sellers if no signals are found.
    Items already in the basket are excluded from all recommendations.
    """
    is_b2c = (customer_name == 'SPRZEDAZ FISKALNA' or customer_name not in rfm.index)

    # Build RFM profile for known customers
    rfm_profile = {}
    if not is_b2c and customer_name in rfm.index:
        r = rfm.loc[customer_name]
        rfm_profile = {
            'Recency':   int(r['Recency']),
            'Frequency': int(r['Frequency']),
            'Monetary':  round(float(r['Monetary']), 2),
            'Segment':   r.get('SegmentLabel', 'Unknown')
        }

    recommendations = {
        'Customer':          customer_name,
        'Type':              'B2C (Anonymous/New)' if is_b2c else 'B2B (Known)',
        'RFM_Profile':       rfm_profile,
        'Current_Basket':    current_basket,
        'CF_Recs':           [],
        'AR_Recs':           [],
        'Popularity_Fallback': []
    }

    # --- A. COLLABORATIVE FILTERING (Known B2B only) ---
    if not is_b2c:
        cf_candidates = {}
        for item in current_basket:
            if item in item_similarity_df.columns:
                sims = item_similarity_df[item].sort_values(ascending=False).drop(item)
                for rec_item, score in sims.items():
                    if score > 0 and rec_item not in current_basket:
                        # Keep highest score if the same item appears via multiple basket items
                        if rec_item not in cf_candidates or score > cf_candidates[rec_item]['Score']:
                            cf_candidates[rec_item] = {'Item': rec_item, 'Score': round(score, 2), 'Based_On': item}

        recommendations['CF_Recs'] = sorted(cf_candidates.values(), key=lambda x: x['Score'], reverse=True)[:3]

    # --- B. ASSOCIATION RULES (Both B2B and B2C using respective rule sets) ---
    rules_to_use = b2c_rules if is_b2c else b2b_rules

    if not rules_to_use.empty:
        ar_candidates = {}
        for item in current_basket:
            matches = rules_to_use[rules_to_use['antecedents'] == item]
            for _, row in matches.iterrows():
                rec_item = row['consequents']
                if rec_item not in current_basket and rec_item not in ar_candidates:
                    ar_candidates[rec_item] = {
                        'Item':       rec_item,
                        'Lift':       round(row['lift'], 2),
                        'Confidence': round(row['confidence'], 2),
                        'Based_On':   item
                    }
        recommendations['AR_Recs'] = sorted(ar_candidates.values(), key=lambda x: x['Lift'], reverse=True)[:3]

    # --- C. POPULARITY FALLBACK ---
    if not recommendations['CF_Recs'] and not recommendations['AR_Recs']:
        top_items = [p for p in popularity['ProductName'].tolist() if p not in current_basket]
        recommendations['Popularity_Fallback'] = top_items[:3]

    return recommendations


In [10]:
print("\n--- Testing the Hybrid Pipeline & LLM Prompt ---")

# Test Case 1: A Known Contractor buying a specific item
# (You will need to replace 'TIKKURILLA-PIGMENT CW CZARNY S(SS)' with an actual product from your CSV)
sample_contractor = 'M-ART MAREK BORTNICZUK'
sample_basket = ['TIKKURILLA-PIGMENT CW CZARNY S(SS)'] 

hybrid_results_b2b = get_hybrid_recommendations(sample_contractor, sample_basket)
print(hybrid_results_b2b)


--- Testing the Hybrid Pipeline & LLM Prompt ---
{'Customer': 'M-ART MAREK BORTNICZUK', 'Type': 'B2B (Known)', 'Current_Basket': ['TIKKURILLA-PIGMENT CW CZARNY S(SS)'], 'CF_Recs': [{'Item': 'TIKKURILLA-PIGMENT YX ZOLTY TL. (YO)', 'Score': 0.87, 'Based_On': 'TIKKURILLA-PIGMENT CW CZARNY S(SS)'}, {'Item': 'TIKKURILA OPTIVA 5 BAZA A 9L', 'Score': 0.72, 'Based_On': 'TIKKURILLA-PIGMENT CW CZARNY S(SS)'}], 'AR_Recs': [{'Item': 'TIKKURILLA-PIGMENT YX ZOLTY TL. (YO)', 'Lift': 49.12, 'Confidence': 0.94, 'Based_On': 'TIKKURILLA-PIGMENT CW CZARNY S(SS)'}, {'Item': 'TIKKURILLA-PIGMENT YX ZOLTY TL. (YO)', 'Lift': 46.18, 'Confidence': 0.36, 'Based_On': 'TIKKURILLA-PIGMENT CW CZARNY S(SS)'}], 'Popularity_Fallback': []}


In [ ]:
import os

# ==========================================
# 3. THE LLM PROMPT GENERATOR
# ==========================================
def generate_llm_prompt(hybrid_output):
    """
    Translates raw recommendation data into a structured natural-language prompt
    for an LLM, enriched with the customer's RFM profile.
    """
    rfm_profile = hybrid_output.get('RFM_Profile', {})
    rfm_context = ""
    if rfm_profile:
        rfm_context = (
            f"- Customer History: Last purchase {rfm_profile['Recency']} days ago, "
            f"{rfm_profile['Frequency']} total invoices, "
            f"lifetime spend {rfm_profile['Monetary']:.0f} PLN. "
            f"Segment: {rfm_profile['Segment']}.\n"
        )

    prompt = f"""You are an expert retail marketing assistant. Your job is to translate raw algorithmic recommendation data into a simple, actionable explanation for a store manager.

Here is the data for the current customer interaction:
- Customer Profile: {hybrid_output['Type']}
- Customer Name/ID: {hybrid_output['Customer']}
{rfm_context}- Items currently in their cart: {', '.join(hybrid_output['Current_Basket'])}

Algorithmic Recommendations:"""

    if hybrid_output['AR_Recs']:
        prompt += "\n1. Cross-Selling (Market Basket Analysis):\n"
        for rec in hybrid_output['AR_Recs']:
            prompt += (
                f"   - Suggest '{rec['Item']}' because they are buying '{rec['Based_On']}'. "
                f"(Customers buying this combination are {rec['Lift']}x more likely than average).\n"
            )

    if hybrid_output['CF_Recs']:
        prompt += "\n2. Contractor Purchase Patterns (Collaborative Filtering):\n"
        for rec in hybrid_output['CF_Recs']:
            prompt += (
                f"   - Suggest '{rec['Item']}' because similar contractors who buy "
                f"'{rec['Based_On']}' also frequently buy this (similarity: {rec['Score']}).\n"
            )

    if hybrid_output['Popularity_Fallback']:
        prompt += "\n3. Store Best-Sellers (Fallback):\n"
        prompt += (
            f"   - No specific signals found. Recommend best-sellers: "
            f"{', '.join(hybrid_output['Popularity_Fallback'])}.\n"
        )

    prompt += (
        "\nTask: Write a 2-3 sentence summary for the floor manager. "
        "Explain exactly WHAT to recommend to this customer and WHY, "
        "without using jargon like 'Collaborative Filtering' or 'Statistical Lift'. "
        "Focus on practical business logic."
    )
    return prompt


# ==========================================
# 4. LLM API CALL
# ==========================================
def call_llm(prompt, api_key=None):
    """
    Sends the prompt to OpenAI GPT-4o-mini and returns the generated explanation.
    Set your key via the OPENAI_API_KEY environment variable or pass it directly.
    """
    key = api_key or os.environ.get("OPENAI_API_KEY")
    if not key:
        print("[LLM] No API key found. Set OPENAI_API_KEY as an environment variable.")
        print("[LLM] Returning raw prompt instead:\n")
        print(prompt)
        return None

    try:
        from openai import OpenAI
        client = OpenAI(api_key=key)
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3,
            max_tokens=300,
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"[LLM] API call failed: {e}")
        print("[LLM] Returning raw prompt instead:\n")
        print(prompt)
        return None


# ==========================================
# 5. TEST THE FULL PIPELINE
# ==========================================
print("--- Testing the Hybrid Pipeline & LLM Layer ---")

# Test Case 1: Known B2B Contractor
sample_contractor = 'M-ART MAREK BORTNICZUK'
sample_basket_b2b = ['TIKKURILLA-PIGMENT CW CZARNY S(SS)']

hybrid_b2b   = get_hybrid_recommendations(sample_contractor, sample_basket_b2b)
prompt_b2b   = generate_llm_prompt(hybrid_b2b)

print("\n--- LLM PROMPT (B2B Example) ---")
print(prompt_b2b)
print("\n--- LLM OUTPUT (B2B Example) ---")
response_b2b = call_llm(prompt_b2b)
if response_b2b:
    print(response_b2b)

# Test Case 2: Anonymous Walk-in Customer
sample_basket_b2c = ['TIKKURILLA-PIGMENT CW CZARNY S(SS)']

hybrid_b2c   = get_hybrid_recommendations('SPRZEDAZ FISKALNA', sample_basket_b2c)
prompt_b2c   = generate_llm_prompt(hybrid_b2c)

print("\n--- LLM PROMPT (B2C Example) ---")
print(prompt_b2c)
print("\n--- LLM OUTPUT (B2C Example) ---")
response_b2c = call_llm(prompt_b2c)
if response_b2c:
    print(response_b2c)



--- Testing the Hybrid Pipeline & LLM Prompt ---

--- GENERATED LLM PROMPT (B2B Example) ---

    You are an expert retail marketing assistant. Your job is to translate raw algorithmic recommendation data into a simple, actionable explanation for a store manager.
    
    Here is the data for the current customer interaction:
    - Customer Profile: B2B (Known)
    - Customer Name/ID: M-ART MAREK BORTNICZUK
    - Items currently in their cart: TIKKURILLA-PIGMENT CW CZARNY S(SS)
    
    Algorithmic Recommendations:
    
1. Cross-Selling (Market Basket Analysis):
   - Recommend 'TIKKURILLA-PIGMENT YX ZOLTY TL. (YO)' because they are buying 'TIKKURILLA-PIGMENT CW CZARNY S(SS)'. (Statistical Lift: 49.12x more likely to buy).
   - Recommend 'TIKKURILLA-PIGMENT YX ZOLTY TL. (YO)' because they are buying 'TIKKURILLA-PIGMENT CW CZARNY S(SS)'. (Statistical Lift: 46.18x more likely to buy).

2. Contractor Patterns (Collaborative Filtering):
   - Recommend 'TIKKURILLA-PIGMENT YX ZOLTY TL. (YO)'